In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import anthropic

In [5]:
documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)


[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join eligibility late enrollment"}', call_id='call_B6dyThXnaP5N9q8BI4TJNKTX', name='search', type='function_call', id='fc_085d691b20148888006a30ddbfb8408190ba8f9eb1563eaa88', namespace=None, status='completed')]

In [20]:
json.loads(call.arguments)

{'query': 'join course discovered course can I join eligibility late enrollment'}

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n   

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join. If you want a certificate, you’ll need to submit your project while submissions are still open.'

In [21]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(651, 30)

In [22]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00011565
